# Jiaozi on Kaggle competitions (select → train → predict → submit)

End-to-end run of a Kaggle vision competition through **our own Module 3** selection,
training on the real competition data, then predicting the hidden test set and writing /
submitting a `submission.csv`.

**Before running**
1. Select a **GPU** runtime (Runtime → Change runtime type → GPU).
2. Have a Kaggle account, and **accept the competition rules** on its page (e.g.
   kaggle.com/competitions/cassava-leaf-disease-classification) — otherwise downloads 403.
3. Add Kaggle creds to Colab **Secrets** as `KAGGLE_USERNAME` and `KAGGLE_KEY`
   (or you'll be prompted).
4. Run the cells in order. Module 1 (LLM) is **not** needed here — selection comes
   straight from Module 3.

In [ ]:
# Checkout the repo + install deps
import os, sys, shutil, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/Isso-W/Jiaozi.git'
REPO_REF = 'integration-recommender'
REPO_DIR = Path('/content/Jiaozi')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', 'kaggle>=1.6'], check=True)
print('Jiaozi ready at', REPO_DIR)

## Mount Drive (cache competition data + checkpoints)

Optional but recommended: the competition downloads once to Drive and checkpoints survive
runtime recycling.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_ROOT = '/content/drive/MyDrive/Jiaozi/kaggle'        # competition data cached here
RUNS_ROOT = '/content/drive/MyDrive/Jiaozi/kaggle_runs'   # generated projects + checkpoints
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(RUNS_ROOT, exist_ok=True)
print('data ->', DATA_ROOT)
print('runs ->', RUNS_ROOT)

## Kaggle credentials

Reads `KAGGLE_USERNAME` / `KAGGLE_KEY` from Colab Secrets, else prompts. Writes
`~/.kaggle/kaggle.json` for the Kaggle API.

In [ ]:
import json
from getpass import getpass
from pathlib import Path

try:
    from google.colab import userdata
except Exception:
    userdata = None

def _secret(name):
    if userdata is not None:
        try:
            return (userdata.get(name) or '').strip()
        except Exception:
            return ''
    return ''

username = _secret('KAGGLE_USERNAME') or input('KAGGLE_USERNAME: ').strip()
key = _secret('KAGGLE_KEY') or getpass('KAGGLE_KEY (hidden): ').strip()
os.environ['KAGGLE_USERNAME'] = username
os.environ['KAGGLE_KEY'] = key

kdir = Path.home() / '.kaggle'
kdir.mkdir(parents=True, exist_ok=True)
(kdir / 'kaggle.json').write_text(json.dumps({'username': username, 'key': key}))
(kdir / 'kaggle.json').chmod(0o600)
print('Kaggle credentials set for:', username)
print('Make sure you accepted the competition rules on its Kaggle page (else 403).')

## Select + generate the project (via Module 3)

Downloads the competition, reads its CSV labels, runs **Module 3** to pick the
backbone/head/loss/checkpoint, and generates a real-training project wired to the local
Kaggle data. `model.py` uses a validated template by default (set `LLM_PROVIDER` to use an
LLM instead).

In [ ]:
BENCHMARK_KEY = 'cassava'   # cassava / state_farm / siim_isic / diabetic_retinopathy
PRIORITY = 'balanced'       # speed / accuracy / balanced (balanced favours a finetuneable CNN)
LLM_PROVIDER = None         # e.g. 'openai' to generate model.py with an LLM; None = template

from run_kaggle_benchmark import prepare_project

result = prepare_project(
    BENCHMARK_KEY,
    data_root=DATA_ROOT,
    output_dir=f'{RUNS_ROOT}/{BENCHMARK_KEY}',
    priority=PRIORITY,
    llm_provider=LLM_PROVIDER,
)
PROJECT_DIR = result['module4']['output_dir']
print('\nModule 3 top picks:', [r['backbone'] for r in result['recommendations']])
print('Project generated at:', PROJECT_DIR)

## Train on the competition data (GPU)

Trains the generated project on the real Kaggle train set. Checkpoints go to Drive with
auto-resume, so a disconnect/re-run continues instead of restarting.

In [ ]:
import json, subprocess, sys
from pathlib import Path

EPOCHS = None   # None -> pipeline-recommended epochs

cfg_path = Path(PROJECT_DIR) / 'configs.json'
configs = json.loads(cfg_path.read_text(encoding='utf-8'))
cfg = configs[0]
mc = cfg.get('model_config', cfg)
epochs = EPOCHS or mc.get('recommended_epochs') or cfg.get('recommended_epochs') or 15
backbone = mc.get('backbone') or cfg.get('backbone')

ckpt_dir = f'{RUNS_ROOT}/{BENCHMARK_KEY}/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)
cfg['checkpoint_dir'] = ckpt_dir          # top-level -> read by train_model
cfg['resume_checkpoint'] = 'auto'
cfg_path.write_text(json.dumps(configs, indent=2, ensure_ascii=False), encoding='utf-8')

print(f'Training {backbone} for {epochs} epochs on {BENCHMARK_KEY} ...')
subprocess.run([sys.executable, '-u', 'run.py', '--epochs', str(epochs)], cwd=PROJECT_DIR, text=True).check_returncode()
print('\nTraining done. Checkpoints in', ckpt_dir)

## Predict the hidden test set + submit

Runs inference over the competition test images, writes `submission.csv` in the
sample-submission format, and (if `DO_SUBMIT=True`) submits for a leaderboard score.

In [ ]:
DO_SUBMIT = False   # set True to actually submit to the Kaggle leaderboard
MESSAGE = f'Jiaozi {BENCHMARK_KEY} submission'

from kaggle_submit import predict_and_submit

sub = predict_and_submit(
    BENCHMARK_KEY,
    project_dir=PROJECT_DIR,
    data_root=DATA_ROOT,
    do_submit=DO_SUBMIT,
    message=MESSAGE,
)
print(sub)
print('submission.csv at:', sub['submission'])